In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from pathlib import Path

# Load environment variables
load_dotenv()

api_key = os.getenv("AQUEDUCT_API_KEY", os.getenv("API_KEY"))
if not api_key:
    raise ValueError("API Key is missing. Please set AQUEDUCT_API_KEY or API_KEY in your .env file.")

base_url = "https://aqueduct.ai.datalab.tuwien.ac.at/v1"

# Initialize client
client = OpenAI(
    api_key=api_key,
    base_url=base_url
)

print("✅ Aqueduct OpenAI Client initialized.")

✅ Aqueduct OpenAI Client initialized.


In [3]:
def transcribe_audio(file_path: Path, model_name: str = "whisper-large") -> str:
    """Transcribes a single audio file via the Aqueduct API."""
    try:
        with open(file_path, "rb") as audio_file:
            response = client.audio.transcriptions.create(
                model=model_name,
                file=audio_file
            )
            return response.text
    except Exception as e:
        print(f"\n❌ Error transcribing {file_path.name}: {e}")
        return None

In [4]:
# Configuration
INPUT_FOLDER = Path("D:/Adobe/PP Projects")
OUTPUT_FOLDER = Path("D:/Adobe/PP Projects/transcriptions")

# Valid extensions to look for
SUPPORTED_EXTENSIONS = {".mp3", ".mp4", ".mpeg", ".mpga", ".m4a", ".wav", ".webm"}

# Ensure the output directory exists
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

# Find all files in the input folder
all_files = list(INPUT_FOLDER.iterdir())
audio_files = [f for f in all_files if f.suffix.lower() in SUPPORTED_EXTENSIONS]

if not audio_files:
    print(f"No valid audio files found in '{INPUT_FOLDER.resolve()}'.")
else:
    print(f"Found {len(audio_files)} audio files. Starting batch transcription...\n")

    for idx, audio_path in enumerate(audio_files, 1):
        print(f"[{idx}/{len(audio_files)}] Processing: {audio_path.name}...")

        # Define the output text file path
        output_txt_path = OUTPUT_FOLDER / audio_path.with_suffix('.txt').name

        # Skip if we already transcribed this file
        if output_txt_path.exists():
            print(f"  -> Skipped: '{output_txt_path.name}' already exists.")
            continue

        # Transcribe
        transcription_text = transcribe_audio(audio_path)

        # Save to file if successful
        if transcription_text:
            with open(output_txt_path, "w", encoding="utf-8") as text_file:
                text_file.write(transcription_text)
            print(f"  -> Saved to: {output_txt_path.name}")

    print("\n✅ Batch processing complete!")

Found 1 audio files. Starting batch transcription...

[1/1] Processing: test.mp3...
  -> Saved to: test.txt

✅ Batch processing complete!
